# Landscape-Clustergram Visium-HD

https://www.10xgenomics.com/datasets/visium-hd-cytassist-gene-expression-libraries-of-human-lung-cancer-if

In [4]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [5]:
import numpy as np
import pandas as pd
import celldega as dega
from ipywidgets import Widget

In [6]:
# dataset = 'Visium_HD_Human_Pancreas_binned_outputs'
dataset = 'Visium_HD_Human_Lung_Cancer'

base_path = 'data/visium-hd_data/' + dataset + '/binned_outputs/square_008um/'
landscape_files_path = 'data/landscape_files/' + dataset + '/'

# Viz

In [7]:
server_address = dega.viz.get_local_server()
base_url = f'http://localhost:{server_address}/data/landscape_files/' + dataset

In [8]:
Widget.close_all()

landscape_sst = dega.viz.Landscape(
    technology='Visium-HD',
    base_url=base_url,
    square_tile_size=3,
    height=600
)

/var/folders/nj/jhgl3gz93qjgmsh3qn7y44pw0000gp/T/ipykernel_88282/2271176808.py:3: UserWarning: Transformation matrix not found at http://localhost:63137/data/landscape_files/Visium_HD_Human_Lung_Cancer/micron_to_image_transform.csv. Using identity.
  landscape_sst = dega.viz.Landscape(


In [9]:
df_sig = pd.read_parquet(f'{landscape_files_path}/df_sig.parquet')
meta_gene = pd.read_parquet(f'{landscape_files_path}/meta_gene.parquet')
meta_gene['mean_log1p'] = np.log1p(meta_gene['mean'])

In [10]:
mat = dega.clust.Matrix(data=df_sig, meta_row=meta_gene, row_attr=['mean_log1p'])
mat.filter(by='var', num=5000, axis='row')
mat.filter(by='mean', num=1000, axis='row')
mat.norm(by='total', axis='col')
mat.norm(by='zscore', axis='row')
mat.cluster()
cgm = dega.viz.Clustergram(matrix=mat, width=500, height=500)

In [11]:
dega.viz.landscape_clustergram(landscape=landscape_sst, mat=cgm)

## ChatGPT Tentative Cell Type Predictions

| Cluster | Tentative Identity                      | Key Marker Genes                              | Notes                                                                     |
| ------- | --------------------------------------- | --------------------------------------------- | ------------------------------------------------------------------------- |
| **1**   | Tumor (secretory / neuroendocrine)      | **PAM, CPE, CHGA, SCG2, VGF, CARTPT**         | Strong neuroendocrine & secretory profile. Likely tumor core subtype.     |
| **2**   | Tumor (transcriptionally active)        | **SEC11C, TMED9, AAK1, UCHL1, MYLIP, RABEP1** | Active secretory/metabolic tumor program.                                 |
| **3**   | T cells (effector/IFN+)                 | **TRAC, CXCL9, STAT1, IL32, CD3Z, PSMB9**     | Diffuse, highly activated T cell signature. Likely CD8+ dominated.        |
| **4**   | Tumor (edge / artifact?)                | **BTG2, MAPK4, CSTF3, ANKRD29**               | Similar to clusters 1/2. Border-localized — possible sectioning artifact. |
| **5**   | Tumor (classic epithelial / angiogenic) | **FGA, TFF3, VEGFA, WFDC2, IRS2**             | Likely dominant tumor core. Proliferative & angiogenic.                   |
| **6**   | Endothelial cells (vasculature)         | **VWF, PECAM1, EGFL7, ENPP2, SPARCL1**        | Linear morphology, classic vascular identity.                             |
| **7**   | Fibroblasts / Connective stroma         | **COL1A1, DCN, TIMP1, FN1, BGN, LUM**         | ECM-rich, tumor-adjacent stroma. Possibly perivascular.                   |
| **8**   | Plasma B cells                          | **MZB1, IGKC, XBP1, JCHAIN, PRDX4, FKBP11**   | High Ig gene expression, strong ER stress response.                       |
| **9**   | Macrophages / TAMs                      | **CD68, GPNMB, CTSB, LAMP1, APOE, SPP1**      | Classic tumor-associated macrophage (TAM) profile.                        |
| **10**  | Cycling / proliferative cells           | **SEPTIN2, CCNI, ATP6V1F, DBNL, METTL9**      | Mitotic/metabolic signature. Possibly proliferative edge tumor.           |


In [12]:
dega.viz.clustergram_enrich(cgm)